# 02 · Document Grounding

**목표**: 공개 help.sap.com을 근거로 검색, 주입, 답변을 수행하는 관리형 RAG와 streaming을 관찰합니다.

Grounding은 관련 문서 청크를 먼저 찾고 결과를 prompt에 넣어 근거 없는 답변을 줄입니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #2e7d32; letter-spacing: .02em;">🖥️  UI 실행</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">UI: Custom Repository Grounding</h2>
<p style="margin: 7px 0;">Repository ID를 입력하면 해당 Custom vector 저장소의 문서를 목록으로 보고, .txt/.md를 업로드·삭제한 뒤 즉시 Grounding 질의를 실행할 수 있습니다.</p>
  </div>
</div>


In [3]:
from pathlib import Path
import html
import os
import sys
import gradio as gr

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path[:0] = [str(project_root / 'shared'), str(project_root / '3_gen_ai_hub_services')]
from lab_common.config import load_settings, save_grounding_repository_id
from lab_common.grounding_repository import delete_document, list_documents, upload_text_document
from src.grounding.lab_core import run_vector_grounded_query

manager_settings = load_settings()
default_repository_id = os.getenv('GROUNDING_REPOSITORY_ID', '')

def repository_id_or_error(repository_id: str) -> str:
    value = (repository_id or default_repository_id).strip()
    if not value:
        raise ValueError('Repository ID를 입력하거나 .env에 GROUNDING_REPOSITORY_ID를 설정하세요.')
    return value

def document_id(document: dict) -> str:
    value = document.get('id') or document.get('documentId')
    if value:
        return str(value)
    for item in document.get('metadata', []):
        if item.get('key') == 'id' and item.get('value'):
            return str(item['value'][0])
    return ''

def document_stack(repository_id: str):
    documents = list_documents(manager_settings, repository_id_or_error(repository_id))
    cards, ids = [], []
    for document in documents:
        doc_id = document_id(document)
        if doc_id:
            ids.append(doc_id)
        metadata = ' · '.join(f"{item.get('key')}: {', '.join(map(str, item.get('value', [])))}" for item in document.get('metadata', []))
        chunk_count = len(document.get('chunks', []))
        cards.append(f"<article class='document-card'><strong>{html.escape(doc_id or 'ID 없음')}</strong><small>{html.escape(metadata or 'metadata 없음')}</small><span>{chunk_count} chunks</span></article>")
    empty = "<p class='document-empty'>업로드된 문서가 없습니다.</p>"
    return ''.join(cards) if cards else empty, ids

def save_repository(repository_id: str):
    try:
        repo_id = repository_id_or_error(repository_id)
        save_grounding_repository_id(repo_id)
        return repo_id, 'Repository ID를 루트 .env에 저장했습니다.'
    except Exception as error:
        return repository_id, f'저장 실패: {type(error).__name__}: {error}'

def refresh_documents(repository_id: str):
    try:
        stack, ids = document_stack(repository_id)
        return f'{len(ids)}개 문서를 찾았습니다.', stack, gr.Dropdown(choices=ids, value=None)
    except Exception as error:
        return f'문서 목록 조회 실패: {type(error).__name__}: {error}', '', gr.Dropdown(choices=[], value=None)

def upload_document(repository_id: str, file_path: str | None):
    try:
        if not file_path:
            raise ValueError('업로드할 .txt 또는 .md 파일을 선택하세요.')
        path = Path(file_path)
        if path.suffix.lower() not in {'.txt', '.md'}:
            raise ValueError('현재는 UTF-8 .txt와 .md 파일만 지원합니다.')
        repo_id = repository_id_or_error(repository_id)
        upload_text_document(manager_settings, repo_id, filename=path.name, text=path.read_text(encoding='utf-8'))
        stack, ids = document_stack(repo_id)
        return f'{path.name} 업로드를 요청했습니다. 인덱싱에는 잠시 시간이 걸릴 수 있습니다.', stack, gr.Dropdown(choices=ids, value=None)
    except Exception as error:
        return f'업로드 실패: {type(error).__name__}: {error}', '', gr.Dropdown()

def remove_document(repository_id: str, selected_document_id: str | None):
    try:
        repo_id = repository_id_or_error(repository_id)
        delete_document(manager_settings, repo_id, selected_document_id or '')
        stack, ids = document_stack(repo_id)
        return f'{selected_document_id} 문서를 삭제했습니다.', stack, gr.Dropdown(choices=ids, value=None)
    except Exception as error:
        return f'삭제 실패: {type(error).__name__}: {error}', '', gr.Dropdown()

def run_grounding_ui(repository_id: str, user_query: str) -> str:
    try:
        settings = load_settings()
        repo_id = repository_id_or_error(repository_id)
        return run_vector_grounded_query(user_query, settings, repository_id=repo_id)
    except Exception as error:
        return f'Grounding 호출 실패: {type(error).__name__}: {error}'

ui_css = '''
.document-card { padding: 12px; margin: 8px 0; border: 1px solid #d7e3f4; border-radius: 10px; background: #f8fbff; }
.document-card strong, .document-card small, .document-card span { display: block; }
.document-card strong { overflow: hidden; text-overflow: ellipsis; white-space: nowrap; color: #174ea6; }
.document-card small { margin-top: 5px; color: #5f6368; }
.document-card span { margin-top: 7px; font-size: .82em; color: #1a73e8; }
.document-empty { color: #5f6368; font-size: .92em; }
'''

with gr.Blocks(css=ui_css, title='Custom Repository Grounding') as grounding_ui:
    # 노트북 iframe 폭을 확보하기 위해 기본은 접힌 상태로 시작합니다.
    with gr.Sidebar(open=False, width=280, position='left'):
        gr.Markdown('## Repository')
        repository_input = gr.Textbox(label='Repository ID', value=default_repository_id, placeholder='Grounding Management 카드의 UUID')
        with gr.Row():
            save_repository_button = gr.Button('`.env`에 저장', variant='primary')
            refresh_button = gr.Button('새로고침')
        repository_status = gr.Textbox(label='상태', lines=3, interactive=False)
        gr.Markdown('### Uploaded documents')
        document_list = gr.HTML("<p class='document-empty'>새로고침을 눌러 문서를 불러오세요.</p>")
    gr.Markdown('# Custom Repository Grounding')
    with gr.Tabs():
        with gr.Tab('Upload'):
            gr.Markdown('UTF-8 `.txt` 또는 `.md` 파일을 선택한 저장소에 추가합니다.')
            upload_file = gr.File(label='문서 파일', file_types=['.txt', '.md'], type='filepath')
            upload_button = gr.Button('문서 업로드', variant='primary')
        with gr.Tab('Ground'):
            gr.Markdown('왼쪽 Repository의 문서를 근거로 답변합니다.')
            question = gr.Textbox(label='Question', lines=4, placeholder='문서에 관해 질문하세요.')
            ask_button = gr.Button('Grounded answer 생성', variant='primary')
            answer = gr.Textbox(label='Grounded answer', lines=12)
        with gr.Tab('Delete'):
            gr.Markdown('삭제는 즉시 적용되며 되돌릴 수 없습니다.')
            document_selector = gr.Dropdown(label='삭제할 document ID', choices=[])
            delete_button = gr.Button('선택 문서 삭제', variant='stop')
    save_repository_button.click(save_repository, inputs=repository_input, outputs=[repository_input, repository_status])
    refresh_button.click(refresh_documents, inputs=repository_input, outputs=[repository_status, document_list, document_selector])
    upload_button.click(upload_document, inputs=[repository_input, upload_file], outputs=[repository_status, document_list, document_selector])
    delete_button.click(remove_document, inputs=[repository_input, document_selector], outputs=[repository_status, document_list, document_selector])
    ask_button.click(run_grounding_ui, inputs=[repository_input, question], outputs=answer)

grounding_ui.launch(inline=True, share=False, show_error=True, server_name='127.0.0.1', height=900)


/var/folders/qt/8rdpq03552zcr_s4tn9chglm0000gn/T/ipykernel_85038/1876388820.py:99: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=ui_css, title='Custom Repository Grounding') as grounding_ui:


* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">체크포인트</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;">
<li>UI에서 한 번 실행해 결과를 확인한다.</li>
<li>결과가 기대와 다르면 환경 설정과 모델/저장소 이름을 먼저 점검한다.</li>
<li>아래 코드는 같은 기능의 구성 요소를 자세히 살펴보는 선택 심화 자료다.</li>
</ul>
  </div>
</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 코드로 같은 작업 보기</h2>
<p style="margin: 7px 0;">아래 셀은 UI가 사용하는 구성과 동작을 단계별로 확인하는 자료입니다. 기본 실습에서는 실행하지 않아도 됩니다.</p>
  </div>
</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. Grounding pipeline 구성</h2>
<p style="margin: 7px 0;">DocumentGroundingFilter는 검색 위치를 정합니다. DocumentGroundingPlaceholders는 질문 input과 검색 결과 output을 template에 연결합니다. grounding_response는 Grounding 모듈이 자동으로 채웁니다.</p>
  </div>
</div>


In [ ]:
from pathlib import Path
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path[:0] = [str(project_root / 'shared'), str(project_root / '3_gen_ai_hub_services')]
from lab_common.config import load_settings
from src.grounding.lab_core import run_help_grounded_query, stream_help_grounded_query

settings = load_settings()
print("=" * 56)
print("Step 4 · 공개 그라운딩 + 스트리밍")
print("=" * 56)

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. 근거 기반 질의</h2>
<p style="margin: 7px 0;">service.run에는 질문인 user_query만 전달합니다. 내부적으로 검색, context 주입, LLM 답변이 한 번에 수행됩니다.</p>
  </div>
</div>


In [ ]:
# 1) 그라운딩 질의
print("\n[1] 그라운딩 질의: 'What is Joule?'")
answer = run_help_grounded_query("What is Joule?", settings)
print("  응답:", answer[:300])

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. Streaming</h2>
<p style="margin: 7px 0;">run은 완성된 답을 한 번에 받지만 stream은 생성 조각을 순서대로 받습니다. delta.content는 이번 chunk에서 새로 추가된 텍스트입니다.</p>
  </div>
</div>


In [ ]:
# 2) 스트리밍
print("\n[2] 스트리밍: 'What is SAP AI Core?'")
chunks = 0
print("  ", end="")
for delta in stream_help_grounded_query("What is SAP AI Core?", settings):
    chunks += 1
    print(delta, end="", flush=True)
print(f"\n  (총 {chunks}개 청크 수신)")

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">4. Grounding 전후 비교</h2>
<p style="margin: 7px 0;">LLM만 사용한 답과 같은 질문을 비교해 보세요. template에 Context에 없으면 문서에 없음이라고 답하라는 규칙을 추가하면 근거 범위를 더 명확히 제한할 수 있습니다.</p>
  </div>
</div>


In [ ]:
# 3) Step 1 환각 질문 재질의 → 비교
print("\n[3] 환각 비교: 'How does the data masking of the orchestration service work?'")
answer = run_help_grounded_query("How does the data masking of the orchestration service work?", settings)
print("  응답(그라운딩 있음):", answer[:400])

print("\n✅ Step 4 완료 — 그라운딩/스트리밍 동작. [3]을 Step 1 [3]과 비교하세요.")